In [1]:
import pandas as pd

In [6]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

In [7]:
movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
10324,146684,Cosmic Scrat-tastrophe (2015),Animation|Children|Comedy
10325,146878,Le Grand Restaurant (1966),Comedy
10326,148238,A Very Murray Christmas (2015),Comedy
10327,148626,The Big Short (2015),Drama


In [17]:
ratings

,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523
...,...,...,...,...
105334,668,142488,4.0,1451535844
105335,668,142507,3.5,1451535889
105336,668,143385,4.0,1446388585
105337,668,144976,2.5,1448656898


In [18]:
movie_stats = data.groupby("title").agg({
    "rating": ["mean", "count"]
})

movie_stats.columns = ["avg_rating", "num_ratings"]

In [19]:
popular_movies = movie_stats[movie_stats["num_ratings"] > 100]

In [26]:
import warnings
warnings.filterwarnings("ignore")

In [27]:
movie_name = "Toy Story (1995)"

movie_ratings = user_movie_matrix[movie_name]

similar_movies = user_movie_matrix.corrwith(movie_ratings)

similar_movies = similar_movies.dropna()

similar_df = pd.DataFrame(similar_movies, columns=["correlation"])
similar_df = similar_df.join(movie_stats)

# Apply filter
recommendations = similar_df[
    similar_df["num_ratings"] > 100
].sort_values(by="correlation", ascending=False)

print(recommendations.head(10))

                                                    correlation  avg_rating  \
title                                                                         
Toy Story (1995)                                       1.000000    3.907328   
Toy Story 2 (1999)                                     0.709677    3.913462   
Austin Powers: The Spy Who Shagged Me (1999)           0.580651    3.106838   
Crimson Tide (1995)                                    0.578642    3.822430   
Austin Powers: International Man of Mystery (1997)     0.533061    3.415842   
Bug's Life, A (1998)                                   0.506905    3.607843   
Babe (1995)                                            0.504718    3.616279   
Shakespeare in Love (1998)                             0.498968    3.905660   
Who Framed Roger Rabbit? (1988)                        0.498878    3.569565   
Mrs. Doubtfire (1993)                                  0.477007    3.328313   

                                                   

In [28]:
def recommend(movie_name):
    movie_ratings = user_movie_matrix[movie_name]
    similar_movies = user_movie_matrix.corrwith(movie_ratings)
    similar_movies = similar_movies.dropna()

    similar_df = pd.DataFrame(similar_movies, columns=["correlation"])
    similar_df = similar_df.join(movie_stats)
    

    recommendations = similar_df[
        similar_df["num_ratings"] > 100
    ].sort_values(by="correlation", ascending=False)

    return recommendations.head(10)

recommend("Toy Story (1995)")

,correlation,avg_rating,num_ratings
title,,,
Toy Story (1995),1.000000,3.907328,232
Toy Story 2 (1999),0.709677,3.913462,104
Austin Powers: The Spy Who Shagged Me (1999),0.580651,3.106838,117
Crimson Tide (1995),0.578642,3.822430,107
Austin Powers: International Man of Mystery (1997),0.533061,3.415842,101
"Bug's Life, A (1998)",0.506905,3.607843,102
Babe (1995),0.504718,3.616279,129
Shakespeare in Love (1998),0.498968,3.905660,106
Who Framed Roger Rabbit? (1988),0.498878,3.569565,115


In [30]:
# NOW: Build Content-Based
movies["genres"] = movies["genres"].str.replace("|", " ")

In [32]:
# : TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(movies["genres"])

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

In [35]:
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

In [37]:
def content_recommend(movie_name):
    idx = indices[movie_name]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return movies["title"].iloc[movie_indices]
    
content_recommend("Toy Story (1995)")

1815                                          Antz (1998)
2496                                   Toy Story 2 (1999)
2967       Adventures of Rocky and Bullwinkle, The (2000)
3166                     Emperor's New Groove, The (2000)
3811                                Monsters, Inc. (2001)
6617    DuckTales: The Movie - Treasure of the Lost La...
6997                                     Wild, The (2006)
7382                               Shrek the Third (2007)
7987                       Tale of Despereaux, The (2008)
9215    Asterix and the Vikings (Astérix et les Viking...
Name: title, dtype: object

In [48]:
# Now Final Upgrade: HYBRID SYSTEM

def hybrid_recommend(movie_name):
    
    # --- Collaborative Filtering ---
    collab = recommend(movie_name).reset_index()
    collab = collab[["title", "correlation"]]
    
    # --- Content-Based Filtering ---
    content = content_recommend_scores(movie_name)
    
    # --- Merge both results ---
    hybrid = pd.merge(collab, content, on="title", how="outer")
    
    # --- Handle missing values ---
    hybrid["correlation"] = hybrid["correlation"].fillna(0)
    hybrid["content_score"] = hybrid["content_score"].fillna(0)
    
    # --- Final weighted score ---
    hybrid["final_score"] = (0.6 * hybrid["correlation"]) + (0.4 * hybrid["content_score"])
    
    # --- Sort and return top 10 ---
    return hybrid.sort_values(by="final_score", ascending=False).head(10)

hybrid_recommend("Toy Story (1995)")

,title,correlation,content_score,final_score
16,Toy Story 2 (1999),0.709677,1.0,0.825806
15,Toy Story (1995),1.000000,0.0,0.600000
0,"Adventures of Rocky and Bullwinkle, The (2000)",0.000000,1.0,0.400000
2,Asterix and the Vikings (Astérix et les Viking...,0.000000,1.0,0.400000
1,Antz (1998),0.000000,1.0,0.400000
9,"Emperor's New Groove, The (2000)",0.000000,1.0,0.400000
10,"Monsters, Inc. (2001)",0.000000,1.0,0.400000
13,Shrek the Third (2007),0.000000,1.0,0.400000
14,"Tale of Despereaux, The (2008)",0.000000,1.0,0.400000
8,DuckTales: The Movie - Treasure of the Lost La...,0.000000,1.0,0.400000
